In [ ]:
import pandas as pd
import numpy as np

out_path = "../data/results/"

#Cleaned tables from 02_build_dataset (Best lap, Podium, ClassifiedFlag, Grid already added)
qualis = pd.read_csv(out_path + 'quali_clean.csv')
races = pd.read_csv(out_path + 'races_clean.csv')
qualis.shape, races.shape

In [ ]:
per_race_max = qualis.groupby(['Year', 'Round'])['Best'].transform('max')
qualis['Best'] = qualis['Best'].fillna(per_race_max)

qualis['Best'].isna().sum()


In [286]:

#qualis.insert(loc = qualis.columns.get_loc('Best')+1, column='BestQTime',value=0)
qualis['BestQTime'] = qualis.groupby(by=['Year','Round'])[['Best']].transform('min')
#qualis.insert(loc = qualis.columns.get_loc('Best')+2, column='Relative Difference',value=0)
qualis['Percentage Difference'] = (qualis['Best']-qualis['BestQTime'])*100/qualis['BestQTime']
qualis[['Best','BestQTime','Percentage Difference']]





,Best,BestQTime,Percentage Difference
0,90.558,90.558,0.000000
1,90.681,90.558,0.135825
2,90.687,90.558,0.142450
3,90.921,90.558,0.400848
4,91.048,90.558,0.541090
...,...,...,...
2274,93.449,86.745,7.728399
2275,93.653,86.745,7.963571
2276,93.774,86.745,8.103061
2277,94.509,86.745,8.950372


In [288]:
one_race = (qualis['Year'] == 2026) & (qualis['Round']==2)
t = qualis.loc[one_race, 'Best']
(t-t.min())*100/t.min()
qualis.loc[one_race,'Percentage Difference']

1861    0.000000
1862    0.241137
1863    0.381257
1864    0.395377
1865    0.527894
1866    0.590893
1867    0.878737
1868    1.018856
1869    1.148114
1870    1.230666
1871    1.401199
1872    1.404458
1873    1.601060
1874    1.847628
1875    1.868266
1876    1.613008
1877    2.447211
1878    2.941432
1879    3.409585
1880    3.662669
1881    4.269856
1882    5.259385
Name: Percentage Difference, dtype: float64

In [289]:
def form(races, by, window=5):
    """Average finish over the previous `window` races (shifted, so race N only sees races before it). Resets every season.
    by='Abbreviation' gives driver form, by='TeamName' gives team form."""
    #One row per race for each driver/team. For a team this averages its two cars, so both drivers get the same value
    per_race = races.groupby(by=['Year','Round',by], as_index=False)['Position'].mean()
    per_race = per_race.sort_values(by=['Year','Round'])

    #Shift by one race, then average up to `window` earlier races, inside each season
    per_season = per_race.groupby(by=['Year',by])['Position']
    per_race['Form'] = per_season.transform(lambda s: s.shift().rolling(window, min_periods=1).mean())

    #Put the value back on every row of races, keeping the original index
    merged = races[['Year','Round',by]].merge(per_race[['Year','Round',by,'Form']], on=['Year','Round',by], how='left')
    merged.index = races.index
    return merged['Form']

races['DriverForm'] = form(races, 'Abbreviation')
races['TeamForm'] = form(races, 'TeamName')

#Blanks left = round 1 of each season. Median fill comes later
print(races[['DriverForm','TeamForm']].isna().sum())

#Check: both McLaren drivers share one team form per race
mclaren_check = (races['Year'] == 2026) & (races['Round'].isin([19,20])) & (races['TeamName'] == 'McLaren')
races.loc[mclaren_check, ['Round','Abbreviation','Position','DriverForm','TeamForm']]

DriverForm    114
TeamForm      102
dtype: int64


,Round,Abbreviation,Position,DriverForm,TeamForm
2238,19,NOR,5.0,8.0,7.9
2246,19,PIA,13.0,7.8,7.9
2256,20,NOR,1.0,8.4,8.6
2261,20,PIA,6.0,8.8,8.6


In [290]:
qualis['TeamName'].unique()

array(['Ferrari', 'Red Bull Racing', 'Mercedes', 'Alfa Romeo',
       'Haas F1 Team', 'Alpine', 'AlphaTauri', 'McLaren', 'Williams',
       'Aston Martin', 'RB', 'Kick Sauber', 'Racing Bulls', 'Audi',
       'Cadillac'], dtype=object)

In [291]:
#Gap to teammate: % difference between a driver's Best and their teammate's Best in the same session
#Negative = faster than the teammate, positive = slower. Teammates always have the same car, so this measures the driver
#Needs the blank-Best fill (session max) to have run first, otherwise drivers with no time get NaN here
def gap_to_teammate(df):
    team = df.groupby(['Year', 'Round', 'TeamName'])['Best']
    # Two drivers per team: the teammate's time is the team total minus your own
    teammate_best = (team.transform('sum') - df['Best']) / (team.transform('count') - 1)
    gap = (df['Best'] - teammate_best) * 100 / teammate_best
    # 2024 Australia: Sargeant gave his car to Albon, so Albon had no teammate. 0 = no information either way
    return gap.fillna(0)

qualis['GapToTeammate'] = gap_to_teammate(qualis)

#Check: in each team and session the two gaps should have opposite signs, e.g. 2026 Sao Paulo McLaren and Mercedes
check = (qualis['Year'] == 2026) & (qualis['Round'] == 20) & qualis['TeamName'].isin(['McLaren', 'Mercedes'])
qualis.loc[check, ['TeamName', 'Abbreviation', 'Best', 'GapToTeammate']]


,TeamName,Abbreviation,Best,GapToTeammate
2257,McLaren,NOR,87.470,-3.207958
2258,Mercedes,RUS,86.745,-1.532437
2259,Mercedes,ANT,88.095,1.556286
2268,McLaren,PIA,90.369,3.314279


In [292]:
races.columns.tolist()

['DriverNumber',
 'BroadcastName',
 'Abbreviation',
 'DriverId',
 'TeamName',
 'TeamColor',
 'TeamId',
 'FirstName',
 'LastName',
 'FullName',
 'HeadshotUrl',
 'CountryCode',
 'Position',
 'Podium',
 'ClassifiedPosition',
 'ClassifiedFlag',
 'GridPosition',
 'Q1',
 'Q2',
 'Q3',
 'Time',
 'Status',
 'Points',
 'Laps',
 'Year',
 'Round',
 'EventName',
 'Session',
 'Grid',
 'DriverForm',
 'TeamForm']

In [293]:
driver_form_median = races.loc[races['Year']<2025]['DriverForm'].median()
races['DriverForm'] = races['DriverForm'].fillna(driver_form_median)

team_form_median = races.loc[races['Year']<2025]['TeamForm'].median()
races['TeamForm'] = races['TeamForm'].fillna(team_form_median)

In [294]:
def points_before(races, sprints):
    """Constructor points going into each race and the driver's share of them. Resets every season.
    A sprint runs before the race on the same weekend, so its points already count."""
    sprint_points = sprints.groupby(by=['Year','Round','Abbreviation'], as_index=False)['Points'].sum().rename(columns={'Points':'SprintPoints'})
    per_race = races[['Year','Round','Abbreviation','TeamName','Points']].merge(sprint_points, on=['Year','Round','Abbreviation'], how='left')
    per_race['SprintPoints'] = per_race['SprintPoints'].fillna(0)
    per_race['WeekendPoints'] = per_race['Points'] + per_race['SprintPoints']
    per_race = per_race.sort_values(by=['Year','Round'])

    #Driver: points scored for this team so far, minus this race
    per_race['DriverPoints'] = per_race.groupby(by=['Year','TeamName','Abbreviation'])['WeekendPoints'].cumsum() - per_race['Points']

    #Team: summed per weekend first, so points from a driver who has since left still count
    team = per_race.groupby(by=['Year','Round','TeamName'], as_index=False)[['WeekendPoints','Points']].sum()
    team['ConstructorPoints'] = team.groupby(by=['Year','TeamName'])['WeekendPoints'].cumsum() - team['Points']
    per_race = per_race.merge(team[['Year','Round','TeamName','ConstructorPoints']], on=['Year','Round','TeamName'], how='left')

    #Team on 0 points (always true in round 1): split evenly
    per_race['DriverShare'] = np.where(per_race['ConstructorPoints'] > 0, per_race['DriverPoints'] / per_race['ConstructorPoints'], 0.5)

    merged = races[['Year','Round','Abbreviation']].merge(per_race[['Year','Round','Abbreviation','ConstructorPoints','DriverShare']], on=['Year','Round','Abbreviation'], how='left')
    merged.index = races.index
    return merged[['ConstructorPoints','DriverShare']]

sprints = pd.read_csv(out_path + 'sprint.csv')
races[['ConstructorPoints','DriverShare']] = points_before(races, sprints)

#Reliability: share of this season's earlier races where the driver was classified
per_race = races[['Year','Round','Abbreviation','ClassifiedFlag']].sort_values(by=['Year','Round'])
races['Reliability'] = per_race.groupby(by=['Year','Abbreviation'])['ClassifiedFlag'].transform(lambda s: s.shift().expanding().mean())

reliability_median = races.loc[races['Year']<2025]['Reliability'].median()
races['Reliability'] = races['Reliability'].fillna(reliability_median)

#Check: McLaren go into the 2025 finale on 800 points, and scored 33 there (official final total 833)
mclaren_check = (races['Year'] == 2025) & (races['Round'] == 24) & (races['TeamName'] == 'McLaren')
races.loc[mclaren_check, ['Abbreviation','Points','ConstructorPoints','DriverShare','Reliability']]

,Abbreviation,Points,ConstructorPoints,DriverShare,Reliability
1819,PIA,18.0,800.0,0.49,0.913043
1820,NOR,15.0,800.0,0.51,0.956522


In [ ]:
#Bring the qualifying features onto each race row (drivers who qualified but didn't race, like Stroll at 2025 Spain, drop out)
quali_features = qualis[['Year','Round','Abbreviation','Percentage Difference','GapToTeammate']]
merged = races[['Year','Round','Abbreviation']].merge(quali_features, on=['Year','Round','Abbreviation'], how='left')
merged.index = races.index
races[['Percentage Difference','GapToTeammate']] = merged[['Percentage Difference','GapToTeammate']]

#Check: blanks left over, and one team's values
print(races[['Percentage Difference','GapToTeammate']].isna().sum())
races.loc[(races['Year'] == 2026) & (races['Round'] == 20) & (races['TeamName'] == 'McLaren'), ['Abbreviation','Grid','Percentage Difference','GapToTeammate']]